# TREMORS — Anthropic (Claude) API backend

Same event-waveform workflow as the Ollama example, but driven by `ChatAnthropic` instead of `ChatOpenAI`.

This is the pattern for a **custom Anthropic-compatible endpoint** (an internal gateway, a LiteLLM proxy, etc.): point `base_url` at your endpoint **root** (the client appends `/v1/messages`) and pass your key via `api_key`. Leave `base_url` unset to hit the public Anthropic API directly.

Requires the optional extra: `pip install -e ".[anthropic]"`.

> `TremorsAgent` is provider-agnostic — the `ChatAnthropic` instance flows straight into the same 7-tool deepagents graph. Because the model resolves to the `anthropic` provider, the harness profile registered in `_build_deep_agent` keeps the general-purpose `task` tool disabled, so the tool surface is identical to the Ollama/OpenAI backends.

In [ ]:
# Point at your Anthropic-compatible endpoint and key.
# Prefer environment variables so the key never lands in the notebook:
#   export ANTHROPIC_BASE_URL=https://your-gateway.example.com
#   export ANTHROPIC_API_KEY=sk-ant-...
# (Leave base_url unset / None to hit the public Anthropic API directly.)
#
# NOTE: temperature is deliberately NOT set. Some gateway-hosted models reject the
# parameter outright with a 400 ("temperature is not supported"), so cli.py only
# forwards it for the anthropic backend when you pass --temperature explicitly.
# Add temperature=... here only if you know your model accepts it.

import os
from langchain_anthropic import ChatAnthropic
from tremors import TremorsAgent, approve_all

llm = ChatAnthropic(
    base_url="https://YOUR_URL.COM",
    api_key="sk-YOUR-KEY",
    model="MODELNAME",
)

query = "Find 2 unique events in Northern California from 2016 with magnitude > 5.0. Get waveforms and plot them."

agent = TremorsAgent(llm=llm, output_dir="./temp")
result = agent.run(query, on_interrupt=approve_all)